# 🎨 MangaColor: Entrenamiento en Kaggle con GPU (Tesla T4 / P100)

Este notebook te permite entrenar el modelo de colorización **MangaColor** aprovechando las GPUs gratuitas de 16 GB de VRAM de Kaggle.

### Configuración previa en Kaggle:
1. En el menú lateral derecho **Settings** -> **Accelerator**, selecciona **GPU T4 x2** o **GPU P100**.
2. En **Internet**, asegúrate de que esté activado (**Internet On**).

In [ ]:
# 1. Comprobación de aceleración GPU
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 * 1024 * 1024)
    print('GPU Device:', torch.cuda.get_device_name(0))
    print(f'VRAM total: {vram_gb:.2f} GB')
else:
    print('⚠️ ADVERTENCIA: Activa la GPU en Settings -> Accelerator -> GPU T4 x2 o P100.')

In [ ]:
# 2. Clonar el repositorio e instalar dependencias
import os
if not os.path.exists('manga-colorization-v2'):
    !git clone https://github.com/HichamLL04/manga-colorization-v2.git
%cd manga-colorization-v2
!pip install -q -e .
!pip install -q gdown matplotlib

In [ ]:
# 3. Descargar checkpoint base original (v2generator.zip)
import os
if not os.path.exists('v2generator.zip'):
    print('Descargando checkpoint base v2...')
    !gdown --id 1qmxUEKADkEM4iYLp1fpPLLKnfZ6tcF-t -O v2generator.zip
    print('Descarga completada!')
else:
    print('v2generator.zip ya existe.')

### 4. Preparación de datos
Tienes dos opciones:
* **Opción A (Recomendada):** Subir el dataset ya preparado desde tu PC (la carpeta con `manifest.jsonl`, `inputs/` y `targets/`). No consume tiempo de CPU en Kaggle y empieza a entrenar directamente.
* **Opción B:** Subir imágenes en bruto y dejar que `mangacolor prepare` las procese en Kaggle.

In [ ]:
# 4. Localizar o Preparar el dataset
import os, glob

# Si subiste el dataset ya preparado desde tu PC, lo detecta automáticamente:
manifest_candidates = glob.glob('/kaggle/input/**/manifest.jsonl', recursive=True)

if manifest_candidates:
    MANIFEST_PATH = manifest_candidates[0]
    print(f'✓ Dataset ya preparado detectado en: {MANIFEST_PATH}')
else:
    # Opción B: Preparar desde imágenes en bruto
    DATASET_INPUT = '/kaggle/input/mi-manga-dataset/color'  # <-- Ajusta esta ruta a tus imágenes
    DATASET_OUTPUT = '/kaggle/working/data_prepared'
    if os.path.exists(DATASET_INPUT):
        !python -m mangacolor prepare '{DATASET_INPUT}' '{DATASET_OUTPUT}' --val-fraction 0.1 --seed 42
        MANIFEST_PATH = f'{DATASET_OUTPUT}/manifest.jsonl'
    else:
        MANIFEST_PATH = f'{DATASET_OUTPUT}/manifest.jsonl'
        print(f'No se encontró manifest.jsonl ni la ruta {DATASET_INPUT}.')
        print('Sube tu dataset a Kaggle (Add Input) y ejecuta de nuevo.')


### 5. Configurar y Ejecutar el Entrenamiento
Con 16 GB de VRAM en Kaggle puedes entrenar a mayor resolución que en tu portátil:
* `--size 384` o `512` (mayor resolución y nitidez).
* `--batch-size 4` o `8`.
* `--scope decoder` (afina el decodificador para propagar color y el adaptador de paleta) o `--scope adapter` (rápido, solo paleta).

In [ ]:
# 5. Ejecutar entrenamiento supervisado (Fine-Tuning desde el generador v2)
RUN_DIR = '/kaggle/working/runs/experimento_v2'

!python -m mangacolor train \
    '{MANIFEST_PATH}' \
    '{RUN_DIR}' \
    --checkpoint v2generator.zip \
    --device cuda \
    --size 384 \
    --batch-size 4 \
    --scope decoder \
    --epochs 15 \
    --lr 2e-5


In [ ]:
# 6. Gráfica de pérdidas (Losses)
import json
import matplotlib.pyplot as plt

metrics_path = f'{RUN_DIR}/metrics.jsonl'
if os.path.exists(metrics_path):
    steps, totals, recs, hints = [], [], [], []
    for line in open(metrics_path, encoding='utf-8'):
        row = json.loads(line)
        if 'step' in row:
            steps.append(row['step'])
            totals.append(row.get('total', 0))
            recs.append(row.get('reconstruction', 0))
            hints.append(row.get('hint', 0))
    
    plt.figure(figsize=(10, 4))
    plt.plot(steps, totals, label='Total Loss', color='black', linewidth=1.5)
    plt.plot(steps, recs, label='L1 Reconstruction', color='blue', alpha=0.7)
    plt.plot(steps, hints, label='Hint Loss', color='red', alpha=0.7)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Evolución del entrenamiento')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# 7. Exportar generador final nativo v2
!python -m mangacolor export-v2 '{RUN_DIR}/last.pt' '/kaggle/working/generator_v2_trained.zip'
print('Modelo exportado en /kaggle/working/generator_v2_trained.zip')